# 08 LLM Intent Explanation and Seller Action Layer

## Objective

The previous notebooks created the main lead generation workflow:

- user event simulation;
- intent classification;
- lead scoring;
- recommendation strategy;
- simulated A/B test evaluation.

However, a seller-facing product should not stop at giving users a score or a product recommendation.

For SMB sellers, the more useful question is:

> Which potential customer should I follow up with, why does this customer matter, and what action should I take next?

This notebook adds a lightweight LLM-style seller action layer.

The goal is to convert analytical outputs into practical seller-facing actions, such as:

- sending a limited-time offer;
- offering a discount or bundle;
- highlighting delivery reliability;
- showing customer reviews and social proof;
- providing customer support or return guidance;
- sending a personalised product follow-up.

In a real production environment, this layer could be powered by an LLM. In this project, I use transparent business rules to simulate the decision logic so that the recommendation remains explainable and easy to review.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

# The notebook is inside the notebook/ folder.
# Path("..") points back to the project root.
BASE_DIR = Path("..")

# Existing model and recommendation outputs
FINAL_DATA_DIR = BASE_DIR / "data" / "final"

# New seller action outputs will be saved here
OUTPUT_TABLE_DIR = BASE_DIR / "outputs" / "tables"
OUTPUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", BASE_DIR.resolve())
print("Final data folder:", FINAL_DATA_DIR.resolve())
print("Output table folder:", OUTPUT_TABLE_DIR.resolve())

print("Final data folder exists:", FINAL_DATA_DIR.exists())
print("Output table folder exists:", OUTPUT_TABLE_DIR.exists())

Project root: /Users/raina/Desktop/Github/AI-lead-generation-recommendation-analytics
Final data folder: /Users/raina/Desktop/Github/AI-lead-generation-recommendation-analytics/data/final
Output table folder: /Users/raina/Desktop/Github/AI-lead-generation-recommendation-analytics/outputs/tables
Final data folder exists: True
Output table folder exists: True


In [2]:
# Check files in data/final.
# This is a simple sanity check before loading the required tables.

for file in sorted(FINAL_DATA_DIR.glob("*.csv")):
    print(file.name)

fact_lead_scores.csv
fact_recommendation_experiment.csv
fact_recommendations.csv
fact_reviews_llm.csv
fact_user_events.csv


In [3]:
lead_scores_path = FINAL_DATA_DIR / "fact_lead_scores.csv"
intent_path = FINAL_DATA_DIR / "fact_reviews_llm.csv"
recommendation_path = FINAL_DATA_DIR / "fact_recommendations.csv"

required_files = [
    lead_scores_path,
    intent_path,
    recommendation_path
]

for path in required_files:
    print(path.name, "exists:", path.exists())

fact_lead_scores.csv exists: True
fact_reviews_llm.csv exists: True
fact_recommendations.csv exists: True


In [4]:
lead_scores = pd.read_csv(lead_scores_path)
intent_df = pd.read_csv(intent_path)
recommendations = pd.read_csv(recommendation_path)

print("lead_scores:", lead_scores.shape)
print("intent_df:", intent_df.shape)
print("recommendations:", recommendations.shape)

lead_scores: (112650, 30)
intent_df: (112650, 27)
recommendations: (15000, 17)


In [5]:
display(lead_scores.head())
display(intent_df.head())
display(recommendations.head())

,order_id,user_id,product_id,seller_id,category,order_time,price,price_band,gmv,review_score,review_text,delivery_delay_days,late_delivery_flag,traffic_source,device_type,user_value_segment,total_events,viewed,clicked,added_to_cart,inquired,purchased,sentiment,intent_category,purchase_intent,lead_score,high_intent_flag,model_high_intent_probability,model_lead_score,model_high_intent_prediction
0,00010242fe8c5a6d1ba2dd792cb16214,871766c5855e863f6eccc05f988b23cb,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,cool_stuff,2017-09-13 08:59:02,58.90,Medium,72.19,5.0,"Perfeito, produto entregue antes do combinado.",-9.0,0,Creator Profile,iOS,Low Value,5.0,1.0,1.0,1.0,1.0,1.0,positive,ready_to_purchase,high,100.0,1,1.000000,100.00,1
1,00018f77f2f0320c557190d7a144bdd3,eb28e67c4c0b83846050ddfb8a35d051,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,pet_shop,2017-04-26 10:53:06,239.90,Premium,259.83,4.0,no_text_review,-3.0,0,Search,Web,High Value,3.0,1.0,0.0,0.0,1.0,1.0,positive,price_sensitive,high,100.0,1,0.997691,99.77,1
2,000229ec398224ef6ca0657da4fc703e,3818d81c6709e39d06b2738a8d3a2474,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,furniture_decor,2018-01-14 14:33:31,199.00,Premium,216.87,5.0,Chegou antes do prazo previsto e o produto surpreendeu pela qualidade. Muito satisfatório.,-14.0,0,Search,Web,High Value,5.0,1.0,1.0,1.0,1.0,1.0,positive,delivery_concern,high,100.0,1,0.999137,99.91,1
3,00024acbcdf0a6daa1e931b038114c75,af861d436cfc08b2c2ddefd0ba074622,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,perfumery,2018-08-08 10:00:35,12.99,Low,25.78,4.0,no_text_review,-6.0,0,Live Stream,iOS,Low Value,2.0,1.0,0.0,0.0,0.0,1.0,positive,ready_to_purchase,high,98.0,1,0.989320,98.93,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,64b576fb70d441e8f1b2d7d446e483c5,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,garden_tools,2017-02-04 13:57:51,199.90,Premium,218.04,5.0,Gostei pois veio no prazo determinado .,-16.0,0,Creator Profile,iOS,High Value,5.0,1.0,1.0,1.0,1.0,1.0,positive,delivery_concern,high,100.0,1,0.998073,99.81,1


,order_id,user_id,product_id,seller_id,category,order_time,price,price_band,gmv,review_score,review_text,delivery_delay_days,late_delivery_flag,traffic_source,device_type,user_value_segment,total_events,viewed,clicked,added_to_cart,inquired,purchased,sentiment,intent_category,purchase_intent,lead_score,high_intent_flag
0,00010242fe8c5a6d1ba2dd792cb16214,871766c5855e863f6eccc05f988b23cb,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,cool_stuff,2017-09-13 08:59:02,58.90,Medium,72.19,5.0,"Perfeito, produto entregue antes do combinado.",-9.0,0,Creator Profile,iOS,Low Value,5.0,1.0,1.0,1.0,1.0,1.0,positive,ready_to_purchase,high,100.0,1
1,00018f77f2f0320c557190d7a144bdd3,eb28e67c4c0b83846050ddfb8a35d051,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,pet_shop,2017-04-26 10:53:06,239.90,Premium,259.83,4.0,no_text_review,-3.0,0,Search,Web,High Value,3.0,1.0,0.0,0.0,1.0,1.0,positive,price_sensitive,high,100.0,1
2,000229ec398224ef6ca0657da4fc703e,3818d81c6709e39d06b2738a8d3a2474,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,furniture_decor,2018-01-14 14:33:31,199.00,Premium,216.87,5.0,Chegou antes do prazo previsto e o produto surpreendeu pela qualidade. Muito satisfatório.,-14.0,0,Search,Web,High Value,5.0,1.0,1.0,1.0,1.0,1.0,positive,delivery_concern,high,100.0,1
3,00024acbcdf0a6daa1e931b038114c75,af861d436cfc08b2c2ddefd0ba074622,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,perfumery,2018-08-08 10:00:35,12.99,Low,25.78,4.0,no_text_review,-6.0,0,Live Stream,iOS,Low Value,2.0,1.0,0.0,0.0,0.0,1.0,positive,ready_to_purchase,high,98.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,64b576fb70d441e8f1b2d7d446e483c5,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,garden_tools,2017-02-04 13:57:51,199.90,Premium,218.04,5.0,Gostei pois veio no prazo determinado .,-16.0,0,Creator Profile,iOS,High Value,5.0,1.0,1.0,1.0,1.0,1.0,positive,delivery_concern,high,100.0,1


,user_id,user_value_segment,preferred_category,strategy,recommended_product_id,recommended_seller_id,recommended_category,recommendation_score,product_avg_price,product_avg_review_score,product_high_intent_rate,seller_avg_review_score,seller_late_delivery_rate,user_avg_model_lead_score,user_high_intent_rate,experiment_group,recommendation_date
0,842bff27a9fd73c774aeb526d0f53113,Low Value,bed_bath_table,popularity_based,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.877,3.8415,0.097,99.55,1.0,control,2026-01-01
1,842bff27a9fd73c774aeb526d0f53113,Low Value,bed_bath_table,category_preference,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.877,3.8415,0.097,99.55,1.0,treatment_category,2026-01-01
2,842bff27a9fd73c774aeb526d0f53113,Low Value,bed_bath_table,intent_aware,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,1.0309,88.15,3.9444,0.877,3.8415,0.097,99.55,1.0,treatment_intent,2026-01-01
3,97e5be9a2f1841bc42b966aa2d014d13,Medium Value,industry_commerce_and_business,popularity_based,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.877,3.8415,0.097,99.93,1.0,control,2026-01-01
4,97e5be9a2f1841bc42b966aa2d014d13,Medium Value,industry_commerce_and_business,category_preference,3986c6e32f39114eff1ded07f1cb16fb,744dac408745240a2c2528fb1b6028f3,industry_commerce_and_business,0.2079,748.00,5.0000,1.000,4.5542,0.012,99.93,1.0,treatment_category,2026-01-01


## 2. Inspect Key Columns

Before merging the tables, I first inspect the column names.

This step is important because the seller action layer depends on several key fields:

- customer identifier;
- lead score;
- intent category;
- recommendation score;
- product ID;
- seller ID;
- category;
- seller quality signals.

Instead of assuming every column exists, I check the table structures first and then select the useful fields.

In [7]:
print("Lead score columns:")
print(lead_scores.columns.tolist())

print("\nIntent columns:")
print(intent_df.columns.tolist())

print("\nRecommendation columns:")
print(recommendations.columns.tolist())

Lead score columns:
['order_id', 'user_id', 'product_id', 'seller_id', 'category', 'order_time', 'price', 'price_band', 'gmv', 'review_score', 'review_text', 'delivery_delay_days', 'late_delivery_flag', 'traffic_source', 'device_type', 'user_value_segment', 'total_events', 'viewed', 'clicked', 'added_to_cart', 'inquired', 'purchased', 'sentiment', 'intent_category', 'purchase_intent', 'lead_score', 'high_intent_flag', 'model_high_intent_probability', 'model_lead_score', 'model_high_intent_prediction']

Intent columns:
['order_id', 'user_id', 'product_id', 'seller_id', 'category', 'order_time', 'price', 'price_band', 'gmv', 'review_score', 'review_text', 'delivery_delay_days', 'late_delivery_flag', 'traffic_source', 'device_type', 'user_value_segment', 'total_events', 'viewed', 'clicked', 'added_to_cart', 'inquired', 'purchased', 'sentiment', 'intent_category', 'purchase_intent', 'lead_score', 'high_intent_flag']

Recommendation columns:
['user_id', 'user_value_segment', 'preferred_cate

In [8]:
def keep_cols(df, cols):
    """
    Keep columns only if they exist in the dataframe.
    This avoids breaking the notebook when earlier outputs change slightly.
    """
    return [c for c in cols if c in df.columns]

## 3. Select Useful Fields

For this seller action layer, I only keep columns that are relevant for action generation.

The logic is intentionally simple:

- lead score tells us how important the user is;
- intent category tells us what the user may care about;
- recommendation result tells us what product or seller is being suggested;
- seller quality signals help identify potential delivery or review risks.

In [19]:
lead_cols = [
    "user_id",
    "order_id",
    "product_id",
    "seller_id",
    "category",
    "price",
    "price_band",
    "gmv",
    "review_score",
    "delivery_delay_days",
    "late_delivery_flag",
    "traffic_source",
    "device_type",
    "user_value_segment",
    "total_events",
    "viewed",
    "clicked",
    "added_to_cart",
    "inquired",
    "purchased",
    "sentiment",
    "intent_category",
    "purchase_intent",
    "lead_score",
    "high_intent_flag",
    "model_high_intent_probability",
    "model_lead_score",
    "model_high_intent_prediction"
]

intent_cols = [
    "user_id",
    "intent_category",
    "purchase_intent",
    "sentiment",
    "review_score",
    "review_text",
    "delivery_delay_days",
    "late_delivery_flag",
    "price_band",
    "category",
    "gmv",
    "lead_score",
    "high_intent_flag"

]

rec_cols = [
    "user_id",
    "user_value_segment",
    "preferred_category",
    "strategy",
    "recommended_product_id",
    "recommended_seller_id",
    "recommended_category",
    "recommendation_score",
    "product_avg_price",
    "product_avg_review_score",
    "product_high_intent_rate",
    "seller_avg_review_score",
    "seller_late_delivery_rate",
    "user_avg_model_lead_score",
    "user_high_intent_rate",
    "experiment_group",
    "recommendation_date"
]

lead_base = lead_scores[[c for c in lead_cols if c in lead_scores.columns]].copy()
intent_base = intent_df[[c for c in intent_cols if c in intent_df.columns]].copy()
rec_base = recommendations[[c for c in rec_cols if c in recommendations.columns]].copy()

print("lead_base:", lead_base.shape)
print("intent_base:", intent_base.shape)
print("rec_base:", rec_base.shape)

display(lead_base.head())
display(intent_base.head())
display(rec_base.head())

lead_base: (112650, 28)
intent_base: (112650, 13)
rec_base: (15000, 17)


,user_id,order_id,product_id,seller_id,category,price,price_band,gmv,review_score,delivery_delay_days,late_delivery_flag,traffic_source,device_type,user_value_segment,total_events,viewed,clicked,added_to_cart,inquired,purchased,sentiment,intent_category,purchase_intent,lead_score,high_intent_flag,model_high_intent_probability,model_lead_score,model_high_intent_prediction
0,871766c5855e863f6eccc05f988b23cb,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,cool_stuff,58.90,Medium,72.19,5.0,-9.0,0,Creator Profile,iOS,Low Value,5.0,1.0,1.0,1.0,1.0,1.0,positive,ready_to_purchase,high,100.0,1,1.000000,100.00,1
1,eb28e67c4c0b83846050ddfb8a35d051,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,pet_shop,239.90,Premium,259.83,4.0,-3.0,0,Search,Web,High Value,3.0,1.0,0.0,0.0,1.0,1.0,positive,price_sensitive,high,100.0,1,0.997691,99.77,1
2,3818d81c6709e39d06b2738a8d3a2474,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,furniture_decor,199.00,Premium,216.87,5.0,-14.0,0,Search,Web,High Value,5.0,1.0,1.0,1.0,1.0,1.0,positive,delivery_concern,high,100.0,1,0.999137,99.91,1
3,af861d436cfc08b2c2ddefd0ba074622,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,perfumery,12.99,Low,25.78,4.0,-6.0,0,Live Stream,iOS,Low Value,2.0,1.0,0.0,0.0,0.0,1.0,positive,ready_to_purchase,high,98.0,1,0.989320,98.93,1
4,64b576fb70d441e8f1b2d7d446e483c5,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,garden_tools,199.90,Premium,218.04,5.0,-16.0,0,Creator Profile,iOS,High Value,5.0,1.0,1.0,1.0,1.0,1.0,positive,delivery_concern,high,100.0,1,0.998073,99.81,1


,user_id,intent_category,purchase_intent,sentiment,review_score,review_text,delivery_delay_days,late_delivery_flag,price_band,category,gmv,lead_score,high_intent_flag
0,871766c5855e863f6eccc05f988b23cb,ready_to_purchase,high,positive,5.0,"Perfeito, produto entregue antes do combinado.",-9.0,0,Medium,cool_stuff,72.19,100.0,1
1,eb28e67c4c0b83846050ddfb8a35d051,price_sensitive,high,positive,4.0,no_text_review,-3.0,0,Premium,pet_shop,259.83,100.0,1
2,3818d81c6709e39d06b2738a8d3a2474,delivery_concern,high,positive,5.0,Chegou antes do prazo previsto e o produto surpreendeu pela qualidade. Muito satisfatório.,-14.0,0,Premium,furniture_decor,216.87,100.0,1
3,af861d436cfc08b2c2ddefd0ba074622,ready_to_purchase,high,positive,4.0,no_text_review,-6.0,0,Low,perfumery,25.78,98.0,1
4,64b576fb70d441e8f1b2d7d446e483c5,delivery_concern,high,positive,5.0,Gostei pois veio no prazo determinado .,-16.0,0,Premium,garden_tools,218.04,100.0,1


,user_id,user_value_segment,preferred_category,strategy,recommended_product_id,recommended_seller_id,recommended_category,recommendation_score,product_avg_price,product_avg_review_score,product_high_intent_rate,seller_avg_review_score,seller_late_delivery_rate,user_avg_model_lead_score,user_high_intent_rate,experiment_group,recommendation_date
0,842bff27a9fd73c774aeb526d0f53113,Low Value,bed_bath_table,popularity_based,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.877,3.8415,0.097,99.55,1.0,control,2026-01-01
1,842bff27a9fd73c774aeb526d0f53113,Low Value,bed_bath_table,category_preference,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.877,3.8415,0.097,99.55,1.0,treatment_category,2026-01-01
2,842bff27a9fd73c774aeb526d0f53113,Low Value,bed_bath_table,intent_aware,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,1.0309,88.15,3.9444,0.877,3.8415,0.097,99.55,1.0,treatment_intent,2026-01-01
3,97e5be9a2f1841bc42b966aa2d014d13,Medium Value,industry_commerce_and_business,popularity_based,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.877,3.8415,0.097,99.93,1.0,control,2026-01-01
4,97e5be9a2f1841bc42b966aa2d014d13,Medium Value,industry_commerce_and_business,category_preference,3986c6e32f39114eff1ded07f1cb16fb,744dac408745240a2c2528fb1b6028f3,industry_commerce_and_business,0.2079,748.00,5.0000,1.000,4.5542,0.012,99.93,1.0,treatment_category,2026-01-01


## 4. Prepare Table Keys

The main join key is `customer_unique_id`.

If duplicate user rows exist in the intent table or lead score table, I keep one row per user for this seller action layer. The recommendation table can still contain multiple rows per user because one user may receive multiple recommendation candidates.

In [20]:
# The real user key in this project is user_id.
# Keep one lead record and one intent record per user before merging.

USER_KEY = "user_id"

if USER_KEY not in lead_base.columns:
    raise KeyError("user_id is missing from lead_base")

if USER_KEY not in intent_base.columns:
    raise KeyError("user_id is missing from intent_base")

if USER_KEY not in rec_base.columns:
    raise KeyError("user_id is missing from rec_base")


# For lead scores, keep the strongest lead record per user.
# This makes sense because the seller action layer should focus on the most valuable lead signal.
lead_base = (
    lead_base
    .sort_values("lead_score", ascending=False)
    .drop_duplicates(USER_KEY)
)


# For intent records, keep the highest lead score record as the representative intent.
# The intent table does not have a separate intent_score, so lead_score is a practical proxy here.
intent_base = (
    intent_base
    .sort_values("lead_score", ascending=False)
    .drop_duplicates(USER_KEY)
)


print("lead_base after dedup:", lead_base.shape)
print("intent_base after dedup:", intent_base.shape)
print("rec_base after check:", rec_base.shape)

lead_base after dedup: (95420, 28)
intent_base after dedup: (95420, 13)
rec_base after check: (15000, 17)


In [21]:
# The real user key in this project is user_id.
# Keep one lead record and one intent record per user before merging.

USER_KEY = "user_id"

if USER_KEY not in lead_base.columns:
    raise KeyError("user_id is missing from lead_base")

if USER_KEY not in intent_base.columns:
    raise KeyError("user_id is missing from intent_base")

if USER_KEY not in rec_base.columns:
    raise KeyError("user_id is missing from rec_base")


# For lead scores, keep the strongest lead record per user.
# This works because lead_base always contains lead_score from fact_lead_scores.csv.
lead_sort_col = "lead_score" if "lead_score" in lead_base.columns else None

if lead_sort_col:
    lead_base = (
        lead_base
        .sort_values(lead_sort_col, ascending=False)
        .drop_duplicates(USER_KEY)
    )
else:
    lead_base = lead_base.drop_duplicates(USER_KEY)


# For intent records, use lead_score if available.
# If not, keep the first intent record per user.
# This avoids breaking the notebook when the intent table only contains intent labels.
intent_sort_col = "lead_score" if "lead_score" in intent_base.columns else None

if intent_sort_col:
    intent_base = (
        intent_base
        .sort_values(intent_sort_col, ascending=False)
        .drop_duplicates(USER_KEY)
    )
else:
    intent_base = intent_base.drop_duplicates(USER_KEY)


print("lead_base after dedup:", lead_base.shape)
print("intent_base after dedup:", intent_base.shape)
print("rec_base after check:", rec_base.shape)

print("\nlead_base columns:", lead_base.columns.tolist())
print("\nintent_base columns:", intent_base.columns.tolist())
print("\nrec_base columns:", rec_base.columns.tolist())

lead_base after dedup: (95420, 28)
intent_base after dedup: (95420, 13)
rec_base after check: (15000, 17)

lead_base columns: ['user_id', 'order_id', 'product_id', 'seller_id', 'category', 'price', 'price_band', 'gmv', 'review_score', 'delivery_delay_days', 'late_delivery_flag', 'traffic_source', 'device_type', 'user_value_segment', 'total_events', 'viewed', 'clicked', 'added_to_cart', 'inquired', 'purchased', 'sentiment', 'intent_category', 'purchase_intent', 'lead_score', 'high_intent_flag', 'model_high_intent_probability', 'model_lead_score', 'model_high_intent_prediction']

intent_base columns: ['user_id', 'intent_category', 'purchase_intent', 'sentiment', 'review_score', 'review_text', 'delivery_delay_days', 'late_delivery_flag', 'price_band', 'category', 'gmv', 'lead_score', 'high_intent_flag']

rec_base columns: ['user_id', 'user_value_segment', 'preferred_category', 'strategy', 'recommended_product_id', 'recommended_seller_id', 'recommended_category', 'recommendation_score', 'p

## 5. Merge Lead, Intent and Recommendation Results

The seller action layer needs one working table that combines:

- user lead quality;
- user intent;
- recommended product or seller;
- seller and product quality indicators.

Each row in the merged table represents one possible seller action recommendation.

In this project, `user_id` is used as the user-level join key.

In [22]:
# Merge lead score with intent signals first.
# Some fields exist in both tables, so suffixes help keep the source clear.

user_context = lead_base.merge(
    intent_base,
    on="user_id",
    how="left",
    suffixes=("", "_intent")
)

# Merge recommendation candidates with user context.
seller_action_base = rec_base.merge(
    user_context,
    on="user_id",
    how="left",
    suffixes=("", "_lead")
)

print("user_context:", user_context.shape)
print("seller_action_base:", seller_action_base.shape)

display(seller_action_base.head())

user_context: (95420, 40)
seller_action_base: (15000, 56)


,user_id,user_value_segment,preferred_category,strategy,recommended_product_id,recommended_seller_id,recommended_category,recommendation_score,product_avg_price,product_avg_review_score,product_high_intent_rate,seller_avg_review_score,seller_late_delivery_rate,user_avg_model_lead_score,user_high_intent_rate,experiment_group,recommendation_date,order_id,product_id,seller_id,category,price,price_band,gmv,review_score,delivery_delay_days,late_delivery_flag,traffic_source,device_type,user_value_segment_lead,total_events,viewed,clicked,added_to_cart,inquired,purchased,sentiment,intent_category,purchase_intent,lead_score,high_intent_flag,model_high_intent_probability,model_lead_score,model_high_intent_prediction,intent_category_intent,purchase_intent_intent,sentiment_intent,review_score_intent,review_text,delivery_delay_days_intent,late_delivery_flag_intent,price_band_intent,category_intent,gmv_intent,lead_score_intent,high_intent_flag_intent
0,842bff27a9fd73c774aeb526d0f53113,Low Value,bed_bath_table,popularity_based,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.877,3.8415,0.097,99.55,1.0,control,2026-01-01,65a94ea3c1392f150ad232c92d64c3f8,1ae80eade3400d1468bf84ce7bd6b479,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,51.0,Medium,63.70,5.0,-14.0,0,For You Feed,Android,Low Value,3.0,1.0,1.0,0.0,0.0,1.0,positive,ready_to_purchase,high,100.0,1,0.995518,99.55,1,ready_to_purchase,high,positive,5.0,no_text_review,-14.0,0,Medium,bed_bath_table,63.70,100.0,1
1,842bff27a9fd73c774aeb526d0f53113,Low Value,bed_bath_table,category_preference,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.877,3.8415,0.097,99.55,1.0,treatment_category,2026-01-01,65a94ea3c1392f150ad232c92d64c3f8,1ae80eade3400d1468bf84ce7bd6b479,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,51.0,Medium,63.70,5.0,-14.0,0,For You Feed,Android,Low Value,3.0,1.0,1.0,0.0,0.0,1.0,positive,ready_to_purchase,high,100.0,1,0.995518,99.55,1,ready_to_purchase,high,positive,5.0,no_text_review,-14.0,0,Medium,bed_bath_table,63.70,100.0,1
2,842bff27a9fd73c774aeb526d0f53113,Low Value,bed_bath_table,intent_aware,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,1.0309,88.15,3.9444,0.877,3.8415,0.097,99.55,1.0,treatment_intent,2026-01-01,65a94ea3c1392f150ad232c92d64c3f8,1ae80eade3400d1468bf84ce7bd6b479,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,51.0,Medium,63.70,5.0,-14.0,0,For You Feed,Android,Low Value,3.0,1.0,1.0,0.0,0.0,1.0,positive,ready_to_purchase,high,100.0,1,0.995518,99.55,1,ready_to_purchase,high,positive,5.0,no_text_review,-14.0,0,Medium,bed_bath_table,63.70,100.0,1
3,97e5be9a2f1841bc42b966aa2d014d13,Medium Value,industry_commerce_and_business,popularity_based,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.877,3.8415,0.097,99.93,1.0,control,2026-01-01,f26e384b92781c0afd3009e3da835c21,d04bbac48960ecb7ea311b00ca6e1cb7,0dd184061fb0eaa7ca37932c68ab91c5,industry_commerce_and_business,104.0,High,118.72,5.0,-8.0,0,For You Feed,Android,Medium Value,4.0,1.0,1.0,1.0,0.0,1.0,positive,ready_to_purchase,high,100.0,1,0.999318,99.93,1,ready_to_purchase,high,positive,5.0,no_text_review,-8.0,0,High,industry_commerce_and_business,118.72,100.0,1
4,97e5be9a2f1841bc42b966aa2d014d13,Medium Value,industry_commerce_and_business,category_preference,3986c6e32f39114eff1ded07f1cb16fb,744dac408745240a2c2528fb1b6028f3,industry_commerce_and_business,0.2079,748.00,5.0000,1.000,4.5542,0.012,99.93,1.0,treatment_category,2026-01-01,f26e384b92781c0afd3009e3da835c21,d04bbac48960ecb7ea311b00ca6e1cb7,0dd184061fb0eaa7ca37932c68ab91c5,industry_commerce_and_business,104.0,High,118.72,5.0,-8.0,0,For You Feed,Android,Medium Value,4.0,1.0,1.0,1.0,0.0,1.0,positive,ready_to_purchase,high,100.0,1,0.999318,99.93,1,ready_to_purchase,high,positive,5.0,no_text_review,-8.0,0,High,industry_commerce_and_business,118.72,100.0,1


In [23]:
# Quick missing check after merge.
# A few missing values are acceptable, but user_id and recommendation fields should not be empty.

missing_check = (
    seller_action_base
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

missing_check.columns = ["column", "missing_rate"]

display(missing_check.head(20))

,column,missing_rate
0,delivery_delay_days,0.0196
1,delivery_delay_days_intent,0.0196
2,user_id,0.0000
3,model_high_intent_probability,0.0000
4,viewed,0.0000
5,clicked,0.0000
6,added_to_cart,0.0000
7,inquired,0.0000
8,purchased,0.0000
9,sentiment,0.0000


## 6. Build the Seller Action Logic

This step converts lead and intent signals into seller-facing actions.

The action rules are designed based on typical SMB seller needs:

| User signal | Seller action |
|---|---|
| High lead score + strong purchase intent | Send limited-time offer |
| Price-sensitive intent | Send discount or bundle |
| Delivery concern | Highlight delivery reliability |
| Product quality concern | Show reviews and social proof |
| After-sales concern | Offer customer support |
| Medium intent but still valuable | Send personalised product follow-up |

This is a simplified version of an LLM-enhanced decision layer. The output is not just a label. It also includes a short seller message and a business explanation.

In [43]:
def safe_get(row, col, default=None):
    """
    Read a value from a row safely.
    If the column is missing or the value is empty, return a default value.
    """
    if col in row.index and pd.notna(row[col]):
        return row[col]
    return default


def normalise_text(x):
    """
    Convert text into lowercase format for simple rule matching.
    """
    if pd.isna(x):
        return ""
    return str(x).strip().lower()


def assign_seller_action(row):
    """
    Generate a seller-facing next-best-action recommendation.

    The rule order matters here:
    1. handle negative or support-related intent first;
    2. handle delivery, price and quality concerns;
    3. handle ready-to-purchase users;
    4. handle medium-intent users with personalised follow-up.

    This avoids giving aggressive offers to users with negative or low-intent signals.
    """

    intent = normalise_text(safe_get(row, "intent_category", ""))
    purchase_intent = normalise_text(safe_get(row, "purchase_intent", ""))

    lead_score = float(safe_get(row, "lead_score", 0))
    model_lead_score = float(safe_get(row, "model_lead_score", lead_score))
    user_avg_model_lead_score = float(safe_get(row, "user_avg_model_lead_score", model_lead_score))

    late_delivery_flag = int(safe_get(row, "late_delivery_flag", 0))
    seller_late_rate = safe_get(row, "seller_late_delivery_rate", np.nan)
    seller_review_score = safe_get(row, "seller_avg_review_score", np.nan)
    product_review_score = safe_get(row, "product_avg_review_score", np.nan)

    effective_score = max(lead_score, model_lead_score, user_avg_model_lead_score)

    action_type = "nurture_with_general_content"
    priority = "Medium"
    message = "Send a light product reminder and highlight the main product benefits."
    explanation = (
        "The user does not show a highly specific intent signal. "
        "A softer nurturing message is more suitable than an aggressive sales push."
    )

    # 1. Support or negative intent should not receive a hard-selling message.
    if any(word in intent for word in ["support", "after", "return", "refund", "service"]) or intent == "after_sales_issue":
        action_type = "offer_customer_support"
        priority = "Medium"
        message = "Provide customer support, return guidance, warranty information, or service follow-up."
        explanation = (
            "The user may need service reassurance. "
            "A support-oriented follow-up is more appropriate than a hard selling message."
        )

    elif intent in ["general_negative"]:
        action_type = "nurture_with_general_content"
        priority = "Medium"
        message = "Send a soft trust-building message and avoid aggressive promotion."
        explanation = (
            "The user shows a negative or unclear signal. "
            "The seller should rebuild trust before sending conversion-oriented offers."
        )

    # 2. Delivery concern.
    elif any(word in intent for word in ["delivery", "logistics", "shipping"]) or late_delivery_flag == 1:
        action_type = "highlight_delivery_reliability"
        priority = "High" if effective_score >= 70 else "Medium"
        message = "Highlight estimated delivery time, return policy, and reliable fulfilment information."
        explanation = (
            "The user may be concerned about delivery or fulfilment. "
            "The seller should reduce delivery anxiety before pushing for purchase."
        )

    # 3. Price-sensitive users should receive value-based offers.
    elif any(word in intent for word in ["price", "discount", "cost", "value"]) or intent == "price_sensitive":
        action_type = "send_discount_or_bundle"
        priority = "High" if effective_score >= 70 else "Medium"
        message = "Offer a small discount, free shipping, or a bundle deal to reduce price hesitation."
        explanation = (
            "The user appears to be price-sensitive. "
            "A value-focused offer is more useful than a generic product message."
        )

    # 4. Product quality or trust concern.
    elif any(word in intent for word in ["quality", "review", "trust", "defect"]) or intent == "product_quality_concern":
        action_type = "show_social_proof"
        priority = "Medium"
        message = "Show customer reviews, product details, usage examples, and after-sales support."
        explanation = (
            "The user may need more confidence in product quality. "
            "Reviews and product evidence can help build trust."
        )

    # 5. Ready-to-purchase users should get direct conversion action.
    elif intent == "ready_to_purchase" or purchase_intent == "high":
        action_type = "send_limited_time_offer"
        priority = "High" if effective_score >= 70 else "Medium"
        message = "Send a limited-time offer with a clear call-to-action and a fast checkout link."
        explanation = (
            "The user shows strong purchase intent. "
            "The seller should prioritise direct conversion with a clear offer."
        )

    # 6. Comparison shopping users need differentiation.
    elif intent == "comparison_shopping":
        action_type = "show_social_proof"
        priority = "Medium"
        message = "Highlight product advantages, customer reviews, and seller credibility."
        explanation = (
            "The user may be comparing alternatives. "
            "The seller should focus on differentiation and trust-building."
        )

    # 7. Medium-quality leads with unclear intent.
    elif effective_score >= 65:
        action_type = "personalised_product_follow_up"
        priority = "Medium"
        message = "Recommend a relevant product based on the user's preferred category and recent engagement."
        explanation = (
            "The user has a reasonable lead score, but the intent is not very specific. "
            "A personalised follow-up can help move the user closer to purchase."
        )

    warnings = []

    if pd.notna(seller_late_rate) and seller_late_rate > 0.20:
        warnings.append(
            "Seller late delivery risk is relatively high, so fulfilment reliability should be monitored."
        )

    if pd.notna(seller_review_score) and seller_review_score < 3.5:
        warnings.append(
            "Seller average review score is below the ideal level, so exposure should be used carefully."
        )

    if pd.notna(product_review_score) and product_review_score < 3.5:
        warnings.append(
            "The recommended product has a relatively low review score, so social proof may be needed."
        )

    if warnings:
        explanation = explanation + " " + " ".join(warnings)

    return pd.Series({
        "seller_action_type": action_type,
        "seller_action_priority": priority,
        "seller_action_message": message,
        "llm_style_explanation": explanation
    })

## 7. Generate Seller Action Recommendations

Now I apply the action logic to every recommendation candidate.

The generated fields are:

- `seller_action_type`: the suggested next action;
- `seller_action_priority`: High or Medium priority;
- `seller_action_message`: a short seller-facing message;
- `llm_style_explanation`: a natural-language explanation of the recommendation.

In [44]:
action_result = seller_action_base.apply(assign_seller_action, axis=1)

seller_actions = pd.concat(
    [seller_action_base, action_result],
    axis=1
)

print("seller_actions:", seller_actions.shape)
display(seller_actions.head())

seller_actions: (15000, 60)


,user_id,user_value_segment,preferred_category,strategy,recommended_product_id,recommended_seller_id,recommended_category,recommendation_score,product_avg_price,product_avg_review_score,product_high_intent_rate,seller_avg_review_score,seller_late_delivery_rate,user_avg_model_lead_score,user_high_intent_rate,experiment_group,recommendation_date,order_id,product_id,seller_id,category,price,price_band,gmv,review_score,delivery_delay_days,late_delivery_flag,traffic_source,device_type,user_value_segment_lead,total_events,viewed,clicked,added_to_cart,inquired,purchased,sentiment,intent_category,purchase_intent,lead_score,high_intent_flag,model_high_intent_probability,model_lead_score,model_high_intent_prediction,intent_category_intent,purchase_intent_intent,sentiment_intent,review_score_intent,review_text,delivery_delay_days_intent,late_delivery_flag_intent,price_band_intent,category_intent,gmv_intent,lead_score_intent,high_intent_flag_intent,seller_action_type,seller_action_priority,seller_action_message,llm_style_explanation
0,842bff27a9fd73c774aeb526d0f53113,Low Value,bed_bath_table,popularity_based,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.877,3.8415,0.097,99.55,1.0,control,2026-01-01,65a94ea3c1392f150ad232c92d64c3f8,1ae80eade3400d1468bf84ce7bd6b479,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,51.0,Medium,63.70,5.0,-14.0,0,For You Feed,Android,Low Value,3.0,1.0,1.0,0.0,0.0,1.0,positive,ready_to_purchase,high,100.0,1,0.995518,99.55,1,ready_to_purchase,high,positive,5.0,no_text_review,-14.0,0,Medium,bed_bath_table,63.70,100.0,1,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.
1,842bff27a9fd73c774aeb526d0f53113,Low Value,bed_bath_table,category_preference,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.877,3.8415,0.097,99.55,1.0,treatment_category,2026-01-01,65a94ea3c1392f150ad232c92d64c3f8,1ae80eade3400d1468bf84ce7bd6b479,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,51.0,Medium,63.70,5.0,-14.0,0,For You Feed,Android,Low Value,3.0,1.0,1.0,0.0,0.0,1.0,positive,ready_to_purchase,high,100.0,1,0.995518,99.55,1,ready_to_purchase,high,positive,5.0,no_text_review,-14.0,0,Medium,bed_bath_table,63.70,100.0,1,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.
2,842bff27a9fd73c774aeb526d0f53113,Low Value,bed_bath_table,intent_aware,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,1.0309,88.15,3.9444,0.877,3.8415,0.097,99.55,1.0,treatment_intent,2026-01-01,65a94ea3c1392f150ad232c92d64c3f8,1ae80eade3400d1468bf84ce7bd6b479,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,51.0,Medium,63.70,5.0,-14.0,0,For You Feed,Android,Low Value,3.0,1.0,1.0,0.0,0.0,1.0,positive,ready_to_purchase,high,100.0,1,0.995518,99.55,1,ready_to_purchase,high,positive,5.0,no_text_review,-14.0,0,Medium,bed_bath_table,63.70,100.0,1,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.
3,97e5be9a2f1841bc42b966aa2d014d13,Medium Value,industry_commerce_and_business,popularity_based,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.877,3.8415,0.097,99.93,1.0,control,2026-01-01,f26e384b92781c0afd3009e3da835c21,d04bbac48960ecb7ea311b00ca6e1cb7,0dd184061fb0eaa7ca37932c68ab91c5,industry_commerce_and_business,104.0,High,118.72,5.0,-8.0,0,For You Feed,Android,Medium Value,4.0,1.0,1.0,1.0,0.0,1.0,positive,ready_to_purchase,high,100.0,1,0.999318,99.93,1,ready_to_purchase,high,positive,5.0,no_text_revie

In [45]:
action_check = (
    seller_actions
    .groupby(["seller_action_type", "seller_action_priority"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(action_check)

,seller_action_type,seller_action_priority,count
7,send_limited_time_offer,High,7479
5,send_discount_or_bundle,High,2631
0,highlight_delivery_reliability,High,2364
2,nurture_with_general_content,Medium,792
8,show_social_proof,Medium,639
1,highlight_delivery_reliability,Medium,627
6,send_discount_or_bundle,Medium,177
4,personalised_product_follow_up,Medium,162
3,offer_customer_support,Medium,129


In [46]:
seller_actions["seller_action_type"].value_counts(normalize=True).round(4)

seller_action_type
send_limited_time_offer           0.4986
highlight_delivery_reliability    0.1994
send_discount_or_bundle           0.1872
nurture_with_general_content      0.0528
show_social_proof                 0.0426
personalised_product_follow_up    0.0108
offer_customer_support            0.0086
Name: proportion, dtype: float64

In [54]:
intent_action_check = pd.crosstab(
    seller_actions["intent_category"],
    seller_actions["seller_action_type"]
)

display(intent_action_check)

seller_action_type,highlight_delivery_reliability,nurture_with_general_content,offer_customer_support,personalised_product_follow_up,send_discount_or_bundle,send_limited_time_offer,show_social_proof
intent_category,,,,,,,
after_sales_issue,0,0,129,0,0,0,0
comparison_shopping,0,0,0,0,0,3,0
delivery_concern,2991,0,0,0,0,0,0
general_negative,0,687,0,0,0,0,0
neutral_or_unclear,0,105,0,162,0,453,0
price_sensitive,0,0,0,0,2808,0,0
product_quality_concern,0,0,0,0,0,0,639
ready_to_purchase,0,0,0,0,0,7023,0


In [56]:
intent_action_long = (
    seller_actions
    .groupby(["intent_category", "seller_action_type"])
    .size()
    .reset_index(name="count")
    .sort_values(["intent_category", "count"], ascending=[True, False])
)

display(intent_action_long)

,intent_category,seller_action_type,count
0,after_sales_issue,offer_customer_support,129
1,comparison_shopping,send_limited_time_offer,3
2,delivery_concern,highlight_delivery_reliability,2991
3,general_negative,nurture_with_general_content,687
6,neutral_or_unclear,send_limited_time_offer,453
5,neutral_or_unclear,personalised_product_follow_up,162
4,neutral_or_unclear,nurture_with_general_content,105
7,price_sensitive,send_discount_or_bundle,2808
8,product_quality_concern,show_social_proof,639
9,ready_to_purchase,send_limited_time_offer,7023


In [57]:
intent_action_ratio = pd.crosstab(
    seller_actions["intent_category"],
    seller_actions["seller_action_type"],
    normalize="index"
).round(3)

display(intent_action_ratio)

seller_action_type,highlight_delivery_reliability,nurture_with_general_content,offer_customer_support,personalised_product_follow_up,send_discount_or_bundle,send_limited_time_offer,show_social_proof
intent_category,,,,,,,
after_sales_issue,0.0,0.000,1.0,0.000,0.0,0.000,0.0
comparison_shopping,0.0,0.000,0.0,0.000,0.0,1.000,0.0
delivery_concern,1.0,0.000,0.0,0.000,0.0,0.000,0.0
general_negative,0.0,1.000,0.0,0.000,0.0,0.000,0.0
neutral_or_unclear,0.0,0.146,0.0,0.225,0.0,0.629,0.0
price_sensitive,0.0,0.000,0.0,0.000,1.0,0.000,0.0
product_quality_concern,0.0,0.000,0.0,0.000,0.0,0.000,1.0
ready_to_purchase,0.0,0.000,0.0,0.000,0.0,1.000,0.0


In [58]:
intent_action_long["intent_total"] = (
    intent_action_long
    .groupby("intent_category")["count"]
    .transform("sum")
)

intent_action_long["share_within_intent"] = (
    intent_action_long["count"] / intent_action_long["intent_total"]
).round(3)

display(intent_action_long)

,intent_category,seller_action_type,count,intent_total,share_within_intent
0,after_sales_issue,offer_customer_support,129,129,1.000
1,comparison_shopping,send_limited_time_offer,3,3,1.000
2,delivery_concern,highlight_delivery_reliability,2991,2991,1.000
3,general_negative,nurture_with_general_content,687,687,1.000
6,neutral_or_unclear,send_limited_time_offer,453,720,0.629
5,neutral_or_unclear,personalised_product_follow_up,162,720,0.225
4,neutral_or_unclear,nurture_with_general_content,105,720,0.146
7,price_sensitive,send_discount_or_bundle,2808,2808,1.000
8,product_quality_concern,show_social_proof,639,639,1.000
9,ready_to_purchase,send_limited_time_offer,7023,7023,1.000


## 8. Create the Final Seller Action Table

The final output should be easy to read and easy to use in a dashboard or report.

I keep the most important columns only:

- user ID;
- recommended product and seller;
- recommended category;
- recommendation strategy;
- experiment group;
- intent category;
- lead score;
- seller action;
- seller action priority;
- seller-facing message;
- explanation.

In [59]:
final_cols = [
    "user_id",
    "recommended_product_id",
    "recommended_seller_id",
    "recommended_category",
    "preferred_category",
    "strategy",
    "experiment_group",
    "recommendation_score",
    "product_avg_price",
    "product_avg_review_score",
    "product_high_intent_rate",
    "seller_avg_review_score",
    "seller_late_delivery_rate",
    "user_value_segment",
    "traffic_source",
    "device_type",
    "intent_category",
    "purchase_intent",
    "lead_score",
    "model_lead_score",
    "model_high_intent_probability",
    "high_intent_flag",
    "seller_action_type",
    "seller_action_priority",
    "seller_action_message",
    "llm_style_explanation",
    "recommendation_date"
]

final_cols = [c for c in final_cols if c in seller_actions.columns]

seller_actions_final = seller_actions[final_cols].copy()

print("Final table shape:", seller_actions_final.shape)
print("Final columns:")
print(seller_actions_final.columns.tolist())

display(seller_actions_final.head())

Final table shape: (15000, 27)
Final columns:
['user_id', 'recommended_product_id', 'recommended_seller_id', 'recommended_category', 'preferred_category', 'strategy', 'experiment_group', 'recommendation_score', 'product_avg_price', 'product_avg_review_score', 'product_high_intent_rate', 'seller_avg_review_score', 'seller_late_delivery_rate', 'user_value_segment', 'traffic_source', 'device_type', 'intent_category', 'purchase_intent', 'lead_score', 'model_lead_score', 'model_high_intent_probability', 'high_intent_flag', 'seller_action_type', 'seller_action_priority', 'seller_action_message', 'llm_style_explanation', 'recommendation_date']


,user_id,recommended_product_id,recommended_seller_id,recommended_category,preferred_category,strategy,experiment_group,recommendation_score,product_avg_price,product_avg_review_score,product_high_intent_rate,seller_avg_review_score,seller_late_delivery_rate,user_value_segment,traffic_source,device_type,intent_category,purchase_intent,lead_score,model_lead_score,model_high_intent_probability,high_intent_flag,seller_action_type,seller_action_priority,seller_action_message,llm_style_explanation,recommendation_date
0,842bff27a9fd73c774aeb526d0f53113,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,bed_bath_table,popularity_based,control,0.8569,88.15,3.9444,0.877,3.8415,0.097,Low Value,For You Feed,Android,ready_to_purchase,high,100.0,99.55,0.995518,1,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.,2026-01-01
1,842bff27a9fd73c774aeb526d0f53113,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,bed_bath_table,category_preference,treatment_category,0.8569,88.15,3.9444,0.877,3.8415,0.097,Low Value,For You Feed,Android,ready_to_purchase,high,100.0,99.55,0.995518,1,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.,2026-01-01
2,842bff27a9fd73c774aeb526d0f53113,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,bed_bath_table,intent_aware,treatment_intent,1.0309,88.15,3.9444,0.877,3.8415,0.097,Low Value,For You Feed,Android,ready_to_purchase,high,100.0,99.55,0.995518,1,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.,2026-01-01
3,97e5be9a2f1841bc42b966aa2d014d13,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,industry_commerce_and_business,popularity_based,control,0.8569,88.15,3.9444,0.877,3.8415,0.097,Medium Value,For You Feed,Android,ready_to_purchase,high,100.0,99.93,0.999318,1,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.,2026-01-01
4,97e5be9a2f1841bc42b966aa2d014d13,3986c6e32f39114eff1ded07f1cb16fb,744dac408745240a2c2528fb1b6028f3,industry_commerce_and_business,industry_commerce_and_business,category_preference,treatment_category,0.2079,748.00,5.0000,1.000,4.5542,0.012,Medium Value,For You Feed,Android,ready_to_purchase,high,100.0,99.93,0.999318,1,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.,2026-01-01


In [60]:
# Sort by seller action priority and lead score.
# This makes the output look like a practical seller follow-up list.

priority_rank = {
    "High": 1,
    "Medium": 2,
    "Low": 3
}

seller_actions_final["priority_rank"] = (
    seller_actions_final["seller_action_priority"]
    .map(priority_rank)
    .fillna(9)
)

sort_cols = ["priority_rank"]

if "model_lead_score" in seller_actions_final.columns:
    sort_cols.append("model_lead_score")
elif "lead_score" in seller_actions_final.columns:
    sort_cols.append("lead_score")
else:
    sort_cols.append("recommendation_score")

seller_actions_final = seller_actions_final.sort_values(
    by=sort_cols,
    ascending=[True, False]
)

seller_actions_final = seller_actions_final.drop(columns=["priority_rank"])

display(seller_actions_final.head(20))

,user_id,recommended_product_id,recommended_seller_id,recommended_category,preferred_category,strategy,experiment_group,recommendation_score,product_avg_price,product_avg_review_score,product_high_intent_rate,seller_avg_review_score,seller_late_delivery_rate,user_value_segment,traffic_source,device_type,intent_category,purchase_intent,lead_score,model_lead_score,model_high_intent_probability,high_intent_flag,seller_action_type,seller_action_priority,seller_action_message,llm_style_explanation,recommendation_date
12,6baee3008124f2fd07c61fc8169ad485,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,sports_leisure,popularity_based,control,0.8569,88.15,3.9444,0.8770,3.8415,0.0970,High Value,For You Feed,Android,ready_to_purchase,high,100.0,100.0,1.00000,1,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.,2026-01-01
13,6baee3008124f2fd07c61fc8169ad485,e44f675b60b3a3a2453ec36421e06f0f,218d46b86c1881d022bce9c68a7d4b15,sports_leisure,sports_leisure,category_preference,treatment_category,0.2854,106.54,4.1548,0.9167,4.1774,0.0833,High Value,For You Feed,Android,ready_to_purchase,high,100.0,100.0,1.00000,1,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.,2026-01-01
14,6baee3008124f2fd07c61fc8169ad485,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,furniture_decor,sports_leisure,intent_aware,treatment_intent,0.9824,71.35,4.0538,0.9108,4.0944,0.0645,High Value,For You Feed,Android,ready_to_purchase,high,100.0,100.0,1.00000,1,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.,2026-01-01
30,964751adbadc2423f8fc9d924239fffc,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,consoles_games,popularity_based,control,0.8569,88.15,3.9444,0.8770,3.8415,0.0970,Low Value,For You Feed,Android,ready_to_purchase,high,100.0,100.0,1.00000,1,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.,2026-01-01
31,964751adbadc2423f8fc9d924239fffc,0aabfb375647d9738ad0f7b4ea3653b1,37515688008a7a40ac93e3b2e4ab203f,consoles_games,consoles_games,category_preference,treatment_category,0.3228,24.00,4.2199,0.9155,4.1004,0.0795,Low Value,For You Feed,Android,ready_to_purchase,high,100.0,100.0,1.00000,1,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.,2026-01-01
32,964751adbadc2423f8fc9d924239fffc,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,furniture_decor,consoles_games,intent_aware,treatment_intent,0.9324,71.35,4.0538,0.9108,4.0944,0.0645,Low Value,For You Feed,Android,ready_to_purchase,high,100.0,100.0,1.00000,1,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.,2026-01-01
45,3b89072eb40baa6595e30a2d4ed1f4da,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,food_drink,popularity_based,control,0.8569,88.15,3.9444,0.8770,3.8415,0.0970,Low Value,For You Feed,iOS,ready_to_purchase,high,100.0,100.0,1.00000,1,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should priori

## 9. Save the Seller Action Output

The main output of this notebook is:

`outputs/tables/llm_seller_action_recommendations.csv`

This file can be used as:

- a seller CRM follow-up list;
- a Tableau dashboard data source;
- a product prototype output;
- evidence for explaining how the project connects AI signals with SMB seller actions.

In [61]:
seller_action_output_path = OUTPUT_TABLE_DIR / "llm_seller_action_recommendations.csv"

seller_actions_final.to_csv(seller_action_output_path, index=False)

print("Saved file:")
print(seller_action_output_path)

print("Rows:", seller_actions_final.shape[0])
print("Columns:", seller_actions_final.shape[1])

Saved file:
../outputs/tables/llm_seller_action_recommendations.csv
Rows: 15000
Columns: 27


## 10. Create Seller Action Summary

I also create a summary table to make the result easier to explain in the report.

The summary shows:

- how many recommendations are assigned to each action type;
- how priority is distributed;
- the average lead score for each action group.

This helps check whether the action logic produces a reasonable distribution.

In [62]:
summary_cols = ["seller_action_type", "seller_action_priority"]

agg_dict = {
    "user_id": "count"
}

if "model_lead_score" in seller_actions_final.columns:
    agg_dict["model_lead_score"] = "mean"
elif "lead_score" in seller_actions_final.columns:
    agg_dict["lead_score"] = "mean"

seller_action_summary = (
    seller_actions_final
    .groupby(summary_cols)
    .agg(agg_dict)
    .reset_index()
)

seller_action_summary = seller_action_summary.rename(columns={
    "user_id": "recommendation_count",
    "model_lead_score": "avg_model_lead_score",
    "lead_score": "avg_lead_score"
})

for col in ["avg_model_lead_score", "avg_lead_score"]:
    if col in seller_action_summary.columns:
        seller_action_summary[col] = seller_action_summary[col].round(2)

seller_action_summary = seller_action_summary.sort_values(
    by=["seller_action_priority", "recommendation_count"],
    ascending=[True, False]
)

display(seller_action_summary)

,seller_action_type,seller_action_priority,recommendation_count,avg_model_lead_score
7,send_limited_time_offer,High,7479,99.54
5,send_discount_or_bundle,High,2631,97.77
0,highlight_delivery_reliability,High,2364,98.12
2,nurture_with_general_content,Medium,792,40.25
8,show_social_proof,Medium,639,75.98
1,highlight_delivery_reliability,Medium,627,0.97
6,send_discount_or_bundle,Medium,177,1.60
4,personalised_product_follow_up,Medium,162,1.32
3,offer_customer_support,Medium,129,78.08


In [63]:
seller_action_summary_path = OUTPUT_TABLE_DIR / "llm_seller_action_summary.csv"

seller_action_summary.to_csv(seller_action_summary_path, index=False)

print("Saved file:")
print(seller_action_summary_path)

Saved file:
../outputs/tables/llm_seller_action_summary.csv


## 11. Example Seller-Facing Recommendations

The seller action layer makes the project more product-oriented.

Instead of showing a seller only a raw lead score, the system can provide a practical recommendation:

> This user is a high-priority lead. Send a limited-time offer because the user has a strong lead signal.

This is closer to how an AI-assisted SMB growth product would work. The platform should not expect sellers to interpret raw model outputs by themselves. It should translate data signals into clear and actionable next steps.

In [64]:
example_cols = [
    "user_id",
    "recommended_category",
    "intent_category",
    "purchase_intent",
    "lead_score",
    "model_lead_score",
    "recommendation_score",
    "seller_action_type",
    "seller_action_priority",
    "seller_action_message",
    "llm_style_explanation"
]

example_cols = [c for c in example_cols if c in seller_actions_final.columns]

examples = seller_actions_final[example_cols].head(10)

display(examples)

,user_id,recommended_category,intent_category,purchase_intent,lead_score,model_lead_score,recommendation_score,seller_action_type,seller_action_priority,seller_action_message,llm_style_explanation
12,6baee3008124f2fd07c61fc8169ad485,bed_bath_table,ready_to_purchase,high,100.0,100.0,0.8569,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.
13,6baee3008124f2fd07c61fc8169ad485,sports_leisure,ready_to_purchase,high,100.0,100.0,0.2854,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.
14,6baee3008124f2fd07c61fc8169ad485,furniture_decor,ready_to_purchase,high,100.0,100.0,0.9824,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.
30,964751adbadc2423f8fc9d924239fffc,bed_bath_table,ready_to_purchase,high,100.0,100.0,0.8569,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.
31,964751adbadc2423f8fc9d924239fffc,consoles_games,ready_to_purchase,high,100.0,100.0,0.3228,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.
32,964751adbadc2423f8fc9d924239fffc,furniture_decor,ready_to_purchase,high,100.0,100.0,0.9324,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.
45,3b89072eb40baa6595e30a2d4ed1f4da,bed_bath_table,ready_to_purchase,high,100.0,100.0,0.8569,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.
46,3b89072eb40baa6595e30a2d4ed1f4da,food_drink,ready_to_purchase,high,100.0,100.0,0.2112,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.
47,3b89072eb40baa6595e30a2d4ed1f4da,furniture_decor,ready_to_purchase,high,100.0,100.0,0.9324,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.
51,581c85ba0bfacbdb3f21e5ac6fd2187a,bed_bath_table,ready_to_purchase,high,100.0,100.0,0.8569,send_limited_time_offer,High,Send a limited-time offer with a clear call-to-action and a fast checkout link.,The user shows strong purchase intent. The seller should prioritise direct conversion with a clear offer.


## 12. Business Takeaways

This layer adds three product values.

First, it makes lead scoring actionable. Sellers receive a suggested action instead of only a numerical score.

Second, it improves explainability. Each recommendation includes a natural-language explanation that connects user intent, lead quality and seller action.

Third, it connects the project more directly to an SMB lead generation scenario. The output can support seller follow-up, CRM tasks, campaign suggestions, livestream selling scripts, or dashboard alerts.

In a TikTok-style SMB growth product, this layer could later be extended with real LLM-generated seller scripts, buyer-seller conversation summaries, livestream prompts and personalised product explanations.

In [65]:
print("Notebook completed.")
print("Generated outputs:")
print("-", seller_action_output_path.name)
print("-", seller_action_summary_path.name)

Notebook completed.
Generated outputs:
- llm_seller_action_recommendations.csv
- llm_seller_action_summary.csv
